In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

%load_ext rpy2.ipython
import rpy2.robjects as ro


def install_and_load_smooth():
    """Install and load the smooth package from CRAN if not already installed"""
    ro.r('''
    if (!requireNamespace("smooth", quietly=TRUE)) {
        install.packages("smooth", repos="https://cran.rstudio.com/")
    }
    library(smooth)
    ''')
    print("Smooth package loaded from CRAN")

install_and_load_smooth()

In [ ]:
library(Mcomp)
m1_monthly <- subset(M1, "MONTHLY")

In [ ]:
from smooth.adam_general.core.adam import ADAM

DEFAULT_DATA_DIR = "/home/filtheo/smooth/python/tests/speed_tests/benchmark_data"


def list_available_datasets(data_dir=None):
    """List all available datasets in the data directory."""
    data_dir = Path(data_dir) if data_dir else Path(DEFAULT_DATA_DIR)

    if not data_dir.exists():
        return []

    datasets = []
    for path in data_dir.iterdir():
        if path.is_dir() and (path / "metadata.csv").exists():
            datasets.append(path.name)

    return sorted(datasets)


print("Available datasets:", list_available_datasets(DEFAULT_DATA_DIR))

In [ ]:
def load_and_datetime_index_series(dataset_path, series, h,
                                   train_start_date="2023-01-01"):
    """Load train and test dataframes for a series with a monthly DateTimeIndex.

    Parameters
    ----------
    dataset_path : str
        Path to dataset directory.
    series : str
        Series ID (e.g. "series_0001").
    h : int
        Forecast horizon (length of test set).
    train_start_date : str, optional
        Date string for first train index (default '2023-01-01').
    """
    train_path = dataset_path + "/" + series + "_train.csv"
    test_path = dataset_path + "/" + series + "_test.csv"

    train_df = pd.read_csv(train_path)
    total_length = len(train_df)
    dates = pd.date_range(start=train_start_date, periods=total_length, freq="M")
    train_df["t"] = dates
    end_date = train_df["t"].iloc[-1]
    train_df.set_index("t", inplace=True)

    test_df = pd.read_csv(test_path)
    dates = pd.date_range(start=end_date + pd.DateOffset(months=1), periods=h, freq="M")
    test_df["t"] = dates
    test_df.set_index("t", inplace=True)

    return train_df, test_df

In [ ]:
dataset = "m1_monthly"
dataset_path = DEFAULT_DATA_DIR + "/" + dataset

metadata = pd.read_csv(dataset_path + "/metadata.csv")
series_ids = metadata["series_id"].tolist()
h = metadata["horizon"].unique()[0]
freq = metadata["frequency"].unique()[0]

print(f"Dataset: {dataset}, series: {len(series_ids)}, h: {h}, freq: {freq}")

## 1. Automatic ARIMA selection strategy

This notebook benchmarks **automatic non-seasonal ARIMA models** implemented via `adam()/ADAM()` in R and Python on the **M1 monthly** dataset.

We search over a small grid of ARIMA orders `(p, d, q)` (with `p, q ∈ {0,1,2}` and `d ∈ {0,1}`), fit `NNN` ADAM models with `lags = 1`, and select the specification that minimises an information criterion (AICc when available). The same grid and selection rule are used in R and Python so that we can compare selected orders, fit statistics, and forecasts.